# Setup!

In [ ]:
import os, json, time, hashlib, warnings
from fractions import Fraction

import numpy as np
import pandas as pd
from scipy import stats, signal
from scipy.signal import resample_poly, convolve
import nibabel as nib
import nilearn
from nilearn.image import clean_img
try:
    from nilearn.maskers import NiftiLabelsMasker
except ImportError:                                   # older nilearn
    from nilearn.input_data import NiftiLabelsMasker

PROJECT = ".."

# ---- what to run -------------------------------------------------------------
RUN_AROUSAL = True
RUN_FC      = True
FC_DATASETS = ["EEG_Rest", "Pupil_Rest", "Pupil_Story", "FilmFest", "Sherlock"]

# ---- atlas -------------------------------------------------------------------
ATLAS_NAME = "Shen"   # "Shen" (268 ROIs) or "Schaefer" (Schaefer 200 + 8 Brainnetome subcortical = 208 ROIs)
ATLASES = {"Shen":     ("3_atlases/Shen/shen_2mm_268_parcellation.nii.gz", 268),
           "Schaefer": ("3_atlases/Schaefer/Schaefer200_17N_plus_8ROIs_208Parcels.nii.gz", 208)}

# Here you can either use the Shen atlas (268 ROIs) or the Schaefer Atlas + 8 Brainnetome Subcortical (208 ROIs) which should both be in ../3_atlases.

# ---- speed -------------------------------------------------------------------
USE_TS_CACHE = True
N_JOBS       = 1

ATLAS        = os.path.join(PROJECT, ATLASES[ATLAS_NAME][0])
N_ROIS       = ATLASES[ATLAS_NAME][1]
SFX          = "" if ATLAS_NAME == "Shen" else f"_{ATLAS_NAME}"   # FC file suffix: Shen none, Schaefer "_Schaefer"
TS_CACHE_DIR = os.path.join(PROJECT, "4_results", "1_arousal", "_roi_timeseries_cache", ATLAS_NAME)
FC_ATLAS_LOG = os.path.join(PROJECT, "4_results", "1_arousal", "FC_atlas.json")   # read by b_Classification

_n_labels = len(np.setdiff1d(np.unique(np.asarray(nib.load(ATLAS).dataobj)), [0]))
if _n_labels != N_ROIS:
    raise ValueError(f"{ATLAS} has {_n_labels} labels, expected {N_ROIS} for ATLAS_NAME = '{ATLAS_NAME}'")
print(f"Atlas: {ATLAS_NAME} ({ATLAS}, {N_ROIS} ROIs); FC files end in '{SFX or '(no suffix)'}'")


def inp(rel):
    """Input path (always the real project folder)."""
    return os.path.join(PROJECT, rel)


def out(rel):
    """Output path."""
    return os.path.join(PROJECT, rel)


def save_npy(rel, arr):
    """Save, unless the file already holds exactly this array (then it is left untouched, so a run with the
    other atlas does not re-date the shared arousal files and b does not rebuild cut files for nothing)."""
    path = out(rel)
    arr = np.asarray(arr)
    if os.path.exists(path):
        try:
            old = np.load(path, allow_pickle=False)
            if old.shape == arr.shape and np.array_equal(old, arr, equal_nan=True):
                print(f"= Unchanged {arr.shape} → {path}")
                return
        except Exception:
            pass
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.save(path, arr)
    print(f"✅ Saved {arr.shape} → {path}")

# Functions

In [ ]:
def zscore_data(data, n_runs):
    """Z-score normalization across time points for each run and subject."""
    if n_runs > 1:
        return stats.zscore(data, axis=2, nan_policy='omit')
    else:
        return stats.zscore(data, axis=1, nan_policy='omit')


def interpolate_nans(data, n_runs):
    """Interpolates over NaN values in a time series after Z-scoring."""
    if n_runs > 1:
        n_subj, n_runs, n_TRs = data.shape
        interp_data = np.copy(data)
        for subj in range(n_subj):
            for run in range(n_runs):
                y = data[subj, run, :]
                nans = np.isnan(y)
                x = np.arange(len(y))
                if np.any(nans):
                    y[nans] = np.interp(x[nans], x[~nans], y[~nans])
                interp_data[subj, run, :] = y
    else:
        n_subj, n_TRs = data.shape
        interp_data = np.copy(data)
        for subj in range(n_subj):
            y = data[subj, :]
            nans = np.isnan(y)
            x = np.arange(len(y))
            if np.any(nans):
                y[nans] = np.interp(x[nans], x[~nans], y[~nans])
            interp_data[subj, :] = y
    return interp_data


def resample_hrf(hrf_kernel, TR, max_denominator=100):
    """Resamples the HRF kernel to match the TR length."""
    if TR.is_integer():
        hrf_resampled = resample_poly(hrf_kernel, 1, int(TR))
    else:
        frac = Fraction(TR).limit_denominator(max_denominator)
        num, denom = frac.numerator, frac.denominator
        hrf_resampled = resample_poly(hrf_kernel, denom, num)
    return hrf_resampled / np.sum(hrf_resampled)


def convolve_hrf(time_series, hrf_kernel, TR, n_runs, do_convolve=True):
    """Convolves a time-series with the HRF kernel separately for each subject and run."""
    if not do_convolve:
        return time_series
    hrf_resampled = resample_hrf(hrf_kernel, TR)
    if n_runs > 1:
        if time_series.ndim != 3:
            raise ValueError(f"Expected shape (n_subj, n_runs, n_TRs) for multiple runs, got {time_series.shape}")
        n_subj, n_runs, n_TRs = time_series.shape
    else:
        if time_series.ndim == 2:
            time_series = time_series[:, np.newaxis, :]
        elif time_series.ndim != 3:
            raise ValueError(f"Expected shape (n_subj, n_TRs) or (n_subj, n_runs, n_TRs), got {time_series.shape}")
        n_subj, _, n_TRs = time_series.shape
    convolved_signal = np.zeros_like(time_series)
    for subj in range(n_subj):
        for run in range(n_runs):
            convolved_signal[subj, run, :] = convolve(time_series[subj, run, :], hrf_resampled, mode='full')[:n_TRs]
    if n_runs == 1:
        convolved_signal = convolved_signal[:, 0, :]
    return convolved_signal


def smooth_data(data, n_runs, TR, smooth_duration=30):
    window_size = max(1, int(smooth_duration / TR))
    gaussian_window = signal.windows.gaussian(window_size, std=window_size / 4)
    gaussian_window /= np.sum(gaussian_window)
    smoothed_data = np.zeros_like(data)
    if n_runs > 1:
        n_subj, n_runs, n_TRs = data.shape
        for subj in range(n_subj):
            for run in range(n_runs):
                smoothed_data[subj, run, :] = np.convolve(data[subj, run, :], gaussian_window, mode='same')
    else:
        n_subj, n_TRs = data.shape
        for subj in range(n_subj):
            smoothed_data[subj, :] = np.convolve(data[subj, :], gaussian_window, mode='same')
    return smoothed_data


def trim_edges(data, trim_size, n_runs):
    if n_runs > 1:
        return data[:, :, trim_size:-trim_size]
    return data[:, trim_size:-trim_size]


def process_spectral_slopes_per_run(spectral_slopes, hrf_kernel, TR, smooth_duration=30, z_score=True,
                                    n_runs=2, do_convolve=True, do_trim=True):
    """Full processing pipeline per run: Z-score, interpolate, convolve, smooth, and trim."""
    data = zscore_data(spectral_slopes, n_runs) if z_score else spectral_slopes
    interp_data = interpolate_nans(data, n_runs)
    convolved_data = convolve_hrf(interp_data, hrf_kernel, TR=TR, n_runs=n_runs, do_convolve=do_convolve)
    smoothed_data = smooth_data(convolved_data, n_runs=n_runs, TR=TR, smooth_duration=smooth_duration)
    if do_trim:
        trim_size = max(1, int(smooth_duration / (2 * TR)))
        trimmed_data = trim_edges(smoothed_data, trim_size=trim_size, n_runs=n_runs)
    else:
        trimmed_data = smoothed_data
    if n_runs > 1:
        n_subj, n_runs, n_TRs_trimmed = trimmed_data.shape
        return trimmed_data.reshape(n_subj, n_runs * n_TRs_trimmed)
    return trimmed_data

# Arousal Time Courses

In [ ]:
HRF_KERNEL = np.array([0, 0.000354107958396228, 0.0220818694830938, 0.116001537001027,
                       0.221299059999514, 0.242353095826523, 0.186831750619196,
                       0.113041009515928, 0.0572809597709863, 0.0253492394574127,
                       0.0100814114758446, 0.00367740475539297, 0.00124901102357508,
                       0.000399543113110135, 0])
HRF_KERNEL /= np.sum(HRF_KERNEL)

PUPIL_REST_SUBJECTS = ["1372", "1387", "1393", "1416", "1560", "1582", "1604", "1674", "1700",
                       "1766", "1792", "1811", "1834", "1841", "1859", "1913", "1940", "1978",
                       "2079", "2100", "2106", "2196", "2213", "2216", "2220", "2221", "2226"]
PUPIL_STORY_RUN_SIZES = [524, 449, 329]


def load_pupil_rest():
    data_dir = inp("1_data/2_Pupil_Rest_Arousal/")
    subjects = []
    for subj_id in PUPIL_REST_SUBJECTS:
        subj_data = []
        for run_num in [1, 2]:
            file_path = os.path.join(data_dir, f"sub-pa{subj_id}", f"sub-pa{subj_id}_task-rest_run-0{run_num}_et.tsv")
            if not os.path.exists(file_path):
                print(f"File not found: {file_path}")
                continue
            try:
                subj_data.append(np.loadtxt(file_path))
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
        if len(subj_data) == 2:
            subjects.append(np.stack(subj_data, axis=0))
        else:
            print(f"Subject {subj_id} missing one or more runs, skipping.")
    arr = np.array(subjects)
    print(f"Pupil Rest: {arr.shape[0]} subjects loaded, shape {arr.shape}")
    return arr


def load_pupil_story_averages():
    pupil_dir = inp("1_data/3_Pupil_Story_Arousal/")
    total_samples = sum(PUPIL_STORY_RUN_SIZES)
    runs = [[], [], []]
    for file in sorted(f for f in os.listdir(pupil_dir) if f.endswith(".csv")):
        pupil_values = pd.read_csv(os.path.join(pupil_dir, file), header=0)['pupilSize'].values
        if len(pupil_values) == 1303:
            pupil_values = pupil_values[:-1]
        if len(pupil_values) != total_samples:
            print(f"⚠️ Warning: {file} has {len(pupil_values)} values instead of {total_samples}. Skipping.")
            continue
        for k, part in enumerate(np.split(pupil_values, [PUPIL_STORY_RUN_SIZES[0],
                                                         PUPIL_STORY_RUN_SIZES[0] + PUPIL_STORY_RUN_SIZES[1]])):
            runs[k].append(part)
    avgs = [np.mean(zscore_data(np.array(r), n_runs=1), axis=0).reshape(1, -1) for r in runs]
    print(f"Pupil Story: {len(runs[0])} subjects averaged, run lengths {[a.shape[1] for a in avgs]}")
    return avgs


def event_means(csv_rel):
    df = pd.read_csv(inp(csv_rel), header=None, skiprows=1)
    return df.mean(axis=1, skipna=True)


if RUN_AROUSAL:
    pupil_array = load_pupil_rest()
    save_npy("1_data/2_Pupil_Rest_Arousal/Pupil_Rest_Arousal.npy", pupil_array)

    avg_runs = load_pupil_story_averages()
    eeg_slopes = np.load(inp("1_data/1_EEG_Arousal/SpectralSlopes_05Hz_80Hz.npy"))

    processed_eeg = process_spectral_slopes_per_run(eeg_slopes, HRF_KERNEL, TR=2.1)
    processed_pupil = process_spectral_slopes_per_run(pupil_array, HRF_KERNEL, TR=1) # We use this function for non-EEG data, even though the name suggests otherwise!!
    processed_story = np.concatenate([
        process_spectral_slopes_per_run(a, HRF_KERNEL, TR=1, n_runs=1, z_score=False).squeeze() for a in avg_runs
    ]).reshape(1, -1)

    save_npy("4_results/1_arousal/1_EEG_Rest/Convolved_EEG_Rest_Arousal.npy", processed_eeg)
    save_npy("4_results/1_arousal/2_Pupil_Rest/Convolved_Pupil_Rest_Arousal.npy", processed_pupil)
    save_npy("4_results/1_arousal/3_Pupil_Story/Convolved_Pupil_Story_Arousal.npy", processed_story)
    save_npy("4_results/1_arousal/4_FilmFestival/Filmfest_Event_Arousal.npy",
             event_means("1_data/4_FilmFestival_Arousal/FilmFest_arousal_discrete_allSubj.csv"))
    save_npy("4_results/1_arousal/5_Sherlock/Sherlock_Event_Arousal.npy",
             event_means("1_data/5_Sherlock_Arousal/Sherlock_arousal_discrete_allSubj.csv"))

# Dynamic FC

In [ ]:
CONFOUND_COLUMNS = ["global_signal", "white_matter", "csf",
                    "trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"] # We regress these out, generated from fMRI Prep
MOTION_THRESHOLD_MM = 3.0
ROTATION_THRESHOLD_RAD = 3 * (np.pi / 180)


def gaussian_weights(TR, target_window_seconds=30):
    window_size = max(1, int(target_window_seconds / TR))
    sigma = window_size / 4
    center = window_size // 2
    indices = np.arange(window_size)
    weights = np.exp(-((indices - center) ** 2) / (2 * sigma ** 2))
    weights /= np.sum(weights)
    return weights, center


def _file_sig(path):
    st = os.stat(path)
    return [os.path.abspath(path), st.st_size, st.st_mtime_ns]


def roi_timeseries(fmri_path, confound_matrix):
    """clean_img + atlas masker (the slow step), cached per run. Returns (time_series, was_cached)."""
    key = {"fmri": _file_sig(fmri_path), "atlas": _file_sig(ATLAS), "nilearn": nilearn.__version__,
           "confounds": hashlib.sha1(np.ascontiguousarray(confound_matrix.to_numpy()).tobytes()
                                     + ",".join(confound_matrix.columns).encode()).hexdigest()}
    name = os.path.basename(fmri_path).replace(".nii.gz", "")
    cache_npy = os.path.join(TS_CACHE_DIR, os.path.basename(os.path.dirname(fmri_path)), name + ".npy")
    cache_json = cache_npy[:-4] + ".json"
    if USE_TS_CACHE and os.path.exists(cache_npy) and os.path.exists(cache_json):
        with open(cache_json) as f:
            if json.load(f) == key:
                return np.load(cache_npy), True
    with warnings.catch_warnings():                    # silences nilearn's standardize deprecation messages
        warnings.simplefilter("ignore", FutureWarning)
        warnings.simplefilter("ignore", DeprecationWarning)
        cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confound_matrix)
        time_series = NiftiLabelsMasker(labels_img=ATLAS).fit_transform(cleaned_img)
    if USE_TS_CACHE:
        os.makedirs(os.path.dirname(cache_npy), exist_ok=True)
        np.save(cache_npy, time_series)
        with open(cache_json, "w") as f:
            json.dump(key, f)
    return time_series, False


def sliding_window_fc(time_series, weights, center):
    """Gaussian-weighted sliding-window FC per TR (edge TRs stay NaN). Same for all three sliding datasets:
    the raw window goes to np.cov and the Gaussian weights enter once, through aweights."""
    n_TRs, n_regions = time_series.shape
    iu = np.triu_indices(n_regions, k=1)
    fc_vectors = np.full((n_TRs, (n_regions * (n_regions - 1)) // 2), np.nan)
    for tr in range(center, n_TRs - center):
        ts_window = time_series[tr - center:tr + center]
        effective_weights = weights[: ts_window.shape[0]]
        effective_weights /= np.sum(effective_weights)          # in place, as in the old notebook
        weighted_cov = np.cov(ts_window.T, aweights=effective_weights)
        std_dev = np.sqrt(np.diag(weighted_cov))
        std_dev[std_dev == 0] = 1e-6
        fc_vectors[tr] = (weighted_cov / np.outer(std_dev, std_dev))[iu]
    return fc_vectors


def event_fc(time_series, bounds, end_offset, notes):
    """One FC vector per event (plain correlation over the event's TRs)."""
    n_TRs, n_regions = time_series.shape
    n_features = (n_regions * (n_regions - 1)) // 2
    iu = np.triu_indices(n_regions, k=1)
    vecs = []
    for start_tr, end_tr in bounds:
        start, end = start_tr, end_tr + end_offset
        if start < 0:
            raise ValueError(f"Event starts before time series: {start}-{end}")   # crashed in the old notebook too
        if end > n_TRs:
            notes.append(f"event {start}-{end} truncated to {n_TRs}")
            end = n_TRs
        ts = time_series[start:end]
        if ts.shape[0] < 2:
            notes.append(f"event {start}-{end} has < 2 TRs → NaN")
            vecs.append(np.full(n_features, np.nan))
            continue
        cov = np.cov(ts.T)
        std = np.sqrt(np.diag(cov))
        std[std == 0] = 1e-6
        vecs.append((cov / np.outer(std, std))[iu])
    return np.stack(vecs)


def process_run(job):
    """Everything for one fMRI run. Returns its FC array plus motion info for the progress line."""
    confounds = pd.read_csv(job["confound_path"], sep="\t").reset_index(drop=True)
    notes = []
    if job.get("confound_rows") is not None:                          # Sherlock: keep only the movie TRs
        s, e = job["confound_rows"]
        if confounds.shape[0] < e:
            return dict(job=job, fc=None, notes=["confound file too short → skipped"])
        confounds = confounds.iloc[s:e].reset_index(drop=True)
    max_t = confounds[["trans_x", "trans_y", "trans_z"]].abs().max().max()
    max_r = confounds[["rot_x", "rot_y", "rot_z"]].abs().max().max()

    time_series, cached = roi_timeseries(job["fmri_path"], confounds[CONFOUND_COLUMNS].copy())
    if job["kind"] == "sliding":
        time_series = time_series[job["cut"]:, :]
        fc = sliding_window_fc(time_series, job["weights"], job["center"])
    else:
        if time_series.shape[0] <= job["cut"]:
            return dict(job=job, fc=None, max_t=max_t, max_r=max_r, cached=cached,
                        notes=["not enough TRs after trim → skipped"])
        fc = event_fc(time_series[job["cut"]:], job["bounds"], job["end_offset"], notes)
    return dict(job=job, fc=fc, max_t=max_t, max_r=max_r, cached=cached, notes=notes)


def run_line(res, i, n):
    j = res["job"]
    flags = []
    if res.get("max_t", 0) > MOTION_THRESHOLD_MM:
        flags.append(f"⚠️ MOTION outlier (max translation {res['max_t']:.2f} mm)")
    if res.get("max_r", 0) > ROTATION_THRESHOLD_RAD:
        flags.append(f"⚠️ ROTATION outlier (max rotation {np.degrees(res['max_r']):.2f}°)")
    extra = flags + res["notes"] + (["cached"] if res.get("cached") else [])
    print(f"[{i:>3}/{n}] {j['dataset']:<11} {j['subj']} run-{j['run']}" + ("  |  " + "  |  ".join(extra) if extra else ""))
    return flags

# Dataset Specific Stuff

In [ ]:
def _digits(s):
    return "".join(ch for ch in s if ch.isdigit())


def _events(xlsx_rel, cols):
    df = pd.read_excel(inp(xlsx_rel))
    for c in cols:
        df[c] = df[c].astype("Int64")
    return df


def build_jobs(name):
    """List the runs of one dataset (sorted file order, like the old notebook) with everything needed to process them."""
    if name == "EEG_Rest":
        d = inp("1_data/1_EEG_Rest_fMRI/")
        weights, center = gaussian_weights(TR=2.1)
    elif name in ("Pupil_Rest", "Pupil_Story"):
        d = inp("1_data/2_Pupil_Rest_fMRI/" if name == "Pupil_Rest" else "1_data/3_Pupil_Story_fMRI/")
        weights, center = gaussian_weights(TR=1)
    elif name == "FilmFest":
        d = inp("1_data/4_FilmFestival_fMRI/")
        ev = _events("1_data/4_FilmFestival_Arousal/Filmfest_timestamps.xlsx",
                     ["Movie1_TR_start", "Movie1_TR_end", "Movie2_TR_start", "Movie2_TR_end"])
    elif name == "Sherlock":
        d = inp("1_data/5_Sherlock_fMRI/")
        ev = _events("1_data/5_Sherlock_Arousal/Sherlock_timestamps.xlsx",
                     ["run1_start", "run1_end", "run2_start", "run2_end"])

    jobs = []
    for file in sorted(f for f in os.listdir(d) if f.endswith(".nii.gz")):
        parts = file.split("_")
        subj = parts[0]
        job = dict(dataset=name, subj=subj, fmri_path=os.path.join(d, file))
        if name == "EEG_Rest":
            run = _digits(parts[2])
            conf = f"{subj}_task-rest_run-{run}_desc-confounds_timeseries.tsv"
            job.update(kind="sliding", cut=0, weights=weights, center=center)
        elif name == "Pupil_Rest":
            run = _digits(parts[2])
            conf = f"{subj}_task-rest_run-0{run}_desc-confounds_timeseries.tsv"
            job.update(kind="sliding", cut=10, weights=weights, center=center)
        elif name == "Pupil_Story":
            run = int(_digits([p for p in parts if "story" in p][0]))
            conf = f"{subj}_task-story{run}_desc-confounds_timeseries.tsv"
            if run not in (1, 2, 3):
                raise ValueError(f"No run number for {file}")
            job.update(kind="sliding", cut={1: 2, 2: 3, 3: 3}[run], weights=weights, center=center)
        elif name == "FilmFest":
            run = int([p for p in parts if p.startswith("run") and "-" not in p][0].replace("run", ""))
            conf = f"{subj}_task-movie_run-{run:02d}_desc-confounds_timeseries.tsv"
            cols = ["Movie1_TR_start", "Movie1_TR_end"] if run == 1 else ["Movie2_TR_start", "Movie2_TR_end"]
            job.update(kind="events", cut=5, end_offset=1,                          # inclusive event ends
                       bounds=[(int(s), int(e)) for s, e in ev[cols].dropna().values])
        elif name == "Sherlock":
            run = 1 if "part1" in parts[2] else 2
            conf = f"{subj}_task-sherlockPart{run}_desc-confounds_timeseries.tsv"
            cols = ["run1_start", "run1_end"] if run == 1 else ["run2_start", "run2_end"]
            rows = (32, 952 if run == 1 else 1036) if subj == "sub-04" else (46, 966 if run == 1 else 1050)
            job.update(kind="events", cut=5, end_offset=0, confound_rows=rows,      # exclusive event ends
                       bounds=[(int(s), int(e)) for s, e in ev[cols].dropna().values])
        job.update(run=run, confound_path=os.path.join(d, conf))
        if not os.path.exists(job["fmri_path"]) or not os.path.exists(job["confound_path"]):
            print(f"Missing file for {subj} run-{run}, skipping.")
            continue
        jobs.append(job)
    return jobs


FC_OUTPUTS = {                                         # e.g. EEG_FC_array.npy (Shen), EEG_FC_array_Schaefer.npy
    "EEG_Rest":    f"4_results/1_arousal/1_EEG_Rest/EEG_FC_array{SFX}.npy",
    "Pupil_Rest":  f"4_results/1_arousal/2_Pupil_Rest/Pupil_FC_array{SFX}.npy",
    "Pupil_Story": f"4_results/1_arousal/3_Pupil_Story/Pupil_Story_FC_array{SFX}.npy",
    "FilmFest":    f"4_results/1_arousal/4_FilmFestival/FilmFestival_FC_array{SFX}.npy",
    "Sherlock":    f"4_results/1_arousal/5_Sherlock/Sherlock_FC_array{SFX}.npy",
}
RUNS_PER_SUBJECT = {"EEG_Rest": 2, "Pupil_Rest": 2, "Pupil_Story": 3, "FilmFest": 2, "Sherlock": 2}

# Run DFC (Motion outliers are not automatically excluded, so you can keep note of them and deal with them later)

In [ ]:
def _results(jobs):
    if N_JOBS == 1:
        for j in jobs:
            yield process_run(j)
    else:
        from joblib import Parallel, delayed
        yield from Parallel(n_jobs=N_JOBS, return_as="generator")(delayed(process_run)(j) for j in jobs)


def run_fc_dataset(name):
    t0 = time.time()
    jobs = build_jobs(name)
    k = RUNS_PER_SUBJECT[name]
    if name in ("EEG_Rest", "Pupil_Rest", "Pupil_Story") and len(jobs) % k != 0:
        raise ValueError(f"{name}: {len(jobs)} runs is not a multiple of {k} runs per subject.")
    print(f"\n=== {name}: {len(jobs)} runs ===")

    path = out(FC_OUTPUTS[name])
    tmp = path[:-4] + ".partial.npy"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    big, kept, outliers, subject_runs = None, [], [], []
    for i, res in enumerate(_results(jobs), start=1):
        if run_line(res, i, len(jobs)):
            outliers.append(f"{res['job']['subj']} run-{res['job']['run']}")
        fc = res["fc"]
        if fc is None:
            continue
        if name in ("EEG_Rest", "Pupil_Rest"):            # → (n_subj, 2, n_TRs, n_features), written straight to disk
            if big is None:
                big = np.lib.format.open_memmap(tmp, mode="w+", dtype=fc.dtype,
                                                shape=(len(jobs) // 2, 2) + fc.shape)
            if fc.shape != big.shape[2:]:
                raise ValueError(f"{name}: run {i} has shape {fc.shape}, expected {big.shape[2:]}")
            big[(i - 1) // 2, (i - 1) % 2] = fc
        elif name == "Pupil_Story":                       # trim 15 TRs per run, concatenate the 3 runs per subject
            if fc.shape[0] > 30:
                fc = fc[15:-15, :]
            else:
                print(f"⚠️ Warning: Subject {(i - 1) // 3 + 1} Run {(i - 1) % 3 + 1} has fewer than 30 TRs. Skipping trimming.")
            subject_runs.append(fc)
            if len(subject_runs) == 3:
                full_fc = np.concatenate(subject_runs, axis=0)
                subject_runs = []
                if big is None:
                    big = np.lib.format.open_memmap(tmp, mode="w+", dtype=full_fc.dtype,
                                                    shape=(len(jobs) // 3,) + full_fc.shape)
                if full_fc.shape != big.shape[1:]:
                    raise ValueError(f"{name}: subject {(i - 1) // 3 + 1} has shape {full_fc.shape}, expected {big.shape[1:]}")
                big[(i - 1) // 3] = full_fc
        else:                                             # events: small, keep in memory
            kept.append(fc)

    if big is not None:
        big.flush()
        shape = big.shape
        del big
        os.replace(tmp, path)
        print(f"✅ Saved {shape} → {path}")
    else:
        if len(kept) % 2 != 0:
            raise ValueError("⚠️ Each subject must have exactly 2 runs.")
        final = np.stack([np.concatenate([kept[s], kept[s + 1]], axis=0) for s in range(0, len(kept), 2)])
        np.save(path, final)
        print(f"✅ Saved {final.shape} → {path}")
    log = {}
    if os.path.exists(FC_ATLAS_LOG):
        with open(FC_ATLAS_LOG) as f:
            log = json.load(f)
    log[f"{name}{SFX}"] = ATLAS_NAME                     # e.g. "EEG_Rest" (Shen), "EEG_Rest_Schaefer"
    with open(FC_ATLAS_LOG, "w") as f:
        json.dump(log, f, indent=1)
    print(f"{name}: {len(outliers)} motion/rotation outlier run(s){': ' + ', '.join(outliers) if outliers else ''}"
          f"  ({(time.time() - t0) / 60:.1f} min)")


if RUN_FC:
    for _name in FC_DATASETS:
        run_fc_dataset(_name)